# LSTM Text Generator — Training Notebook



In [ ]:
import os, sys
# Make sure utils.py is importable from here
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.text     import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from utils import clean_text, build_sequences, save_tokenizer, generate_text
print('TF version:', tf.__version__)

## 1 · Fetch the dataset

In [ ]:
import urllib.request

URL = 'https://www.gutenberg.org/files/100/100-0.txt'
DATA_PATH = '../data/raw_text.txt'

os.makedirs('../data',   exist_ok=True)
os.makedirs('../models', exist_ok=True)

if not os.path.exists(DATA_PATH):
    print('Downloading …')
    urllib.request.urlretrieve(URL, DATA_PATH)

with open(DATA_PATH, encoding='utf-8', errors='ignore') as f:
    raw = f.read()

start = raw.find('THE SONNETS')
raw   = raw[start : start + 300_000]
print(f'Corpus length: {len(raw):,} characters')
print(raw[:400])

## 2 · Clean & Tokenise

In [ ]:
corpus = clean_text(raw)
print('Sample:', corpus[:300])

tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])
vocab_size = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {vocab_size:,}')

In [ ]:
SEQ_LENGTH = 30
X, y = build_sequences(tokenizer, corpus, SEQ_LENGTH)
y_cat = to_categorical(y, num_classes=vocab_size)
print(f'X shape: {X.shape}  |  y_cat shape: {y_cat.shape}')

## 3 · Build the LSTM Model

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

def build_model(vocab_size, seq_length):
    model = Sequential([
        Embedding(vocab_size, 100, input_length=seq_length),
        LSTM(150, return_sequences=True),
        Dropout(0.2),
        LSTM(100),
        Dropout(0.2),
        Dense(vocab_size, activation='softmax'),
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model = build_model(vocab_size, SEQ_LENGTH)
model.summary()

## 4 · Train

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

MODEL_PATH = '../models/lstm_model.keras'

callbacks = [
    EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='loss'),
]

history = model.fit(
    X, y_cat,
    epochs=50,
    batch_size=128,
    callbacks=callbacks,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss']);  ax1.set_title('Loss')
ax2.plot(history.history['accuracy']); ax2.set_title('Accuracy')
plt.tight_layout()
plt.savefig('../data/training_curves.png', dpi=120)
plt.show()

## 5 · Save tokenizer & quick test

In [ ]:
save_tokenizer(tokenizer, '../models/tokenizer.pkl')
print('Tokenizer saved.')

In [ ]:
seed  = 'to be or not to be'
result = generate_text(model, tokenizer, seed, next_words=30, seq_length=SEQ_LENGTH, temperature=1.0)
print(result)